In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_absolute_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

In [ ]:
train = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/train.csv',header=None)
test = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/test.csv',header=None)
labels = pd.read_csv('/kaggle/input/data-science-london-scikit-learn/trainLabels.csv',header=None)

In [ ]:
train.head()

In [ ]:
test.head()

In [ ]:
labels.head()

In [ ]:
train.info()

In [ ]:
test.info()

In [ ]:
X = train
y = np.ravel(labels)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
#normalizing the dataset
normalised = MinMaxScaler(feature_range=(-1,1)).fit(X_train)
X_train_norm = normalised.transform(X_train)
X_test_norm = normalised.transform(X_test)

In [ ]:
#loading Models
#1st being logistic regression
LR = LogisticRegression(penalty='l2',C=1.0)
LR.fit(X_train_norm,y_train)
reg_pred = LR.predict(X_test_norm)

reg_score = accuracy_score(y_test,reg_pred)
print(f'accuracy score is {reg_score}')

In [ ]:
#Grid search and Decision tree
param_grid = [{
    'criterion':['gini','entropy','log_loss'],
    'max_depth':[3,5,7,9,11,13],
    'min_samples_split':[3,5,7,9,11,13]
}]

des = DecisionTreeClassifier()
clf = GridSearchCV(des,param_grid)
clf.fit(X_train_norm,y_train)
pred_tree = clf.predict(X_test_norm)

des_accuracy = accuracy_score(y_test,pred_tree)

print(f'decision tree classifier {des_accuracy}')
print(f'best parameters {clf.best_params_}')

In [ ]:
RF = RandomForestClassifier(n_estimators=1000,criterion='gini',max_depth=11)
RF.fit(X_train_norm,y_train)
y_pred = RF.predict(X_test_norm)
rf_acc = accuracy_score(y_test,y_pred)

print(f'Random forest classifier score {rf_acc}')

In [ ]:
#SVC Classifier Initialization
svc = SVC()
svc.fit(X_train_norm,y_train)
svc_pred = svc.predict(X_test_norm)
svc_score = accuracy_score(y_test,y_pred)
print(f'SVC classifier score {svc_score}')

In [ ]:
#Gradient Boosting Initialisation
gradient = GradientBoostingClassifier(loss='log_loss',learning_rate=0.1,n_estimators=1000)
gradient.fit(X_train_norm,y_train)
grad_pred = gradient.predict(X_test_norm)
grad_acc = accuracy_score(y_test,y_pred)
print(f"gradient boosting classifier score {grad_acc}")

In [ ]:
#using catboost classifier
from catboost import CatBoostClassifier

cat = CatBoostClassifier(iterations=100,depth=7,learning_rate=0.001,verbose=0,loss_function='Logloss',random_seed=42)
cat.fit(X,y)
cat_pred = pd.Series(cat.predict(test))
#cat_acc = accuracy_score(y_test,y_pred)
#print(f'catboost classifier score {cat_acc}')

In [ ]:
cat_pred

In [ ]:
submission = pd.DataFrame({'Solution':cat_pred,'Id':[i for i in range(1,9001)]})
ans = submission.to_csv('sub.csv',index=False)
submission